# Materials science workflows with Oganesson

Follow an atomistic study from structure preparation to analysis, machine-learning descriptors and optional simulations. Examples use ideal Cu, a substitutional alloy and the bundled MoS2 structure. They demonstrate methods, not validated materials predictions.

See the [README](README.md#installation) for installation. Open this notebook from the repository root, using a kernel in the Oganesson environment. Basic Python and familiarity with crystal cells and coordinates are assumed.

## Learning route

1. [Set up and inspect a crystal](#setup)
2. [Prepare alloys, surfaces and defects](#preparation)
3. [Calculate RDF and XRD](#analysis)
4. [Build descriptors and a feature dataset](#descriptors)
5. [Relax structures and prepare NEB images](#relaxation)
6. [Run molecular dynamics](#dynamics)
7. [Search structures with a genetic algorithm](#search)
8. [Extract VASP data and construct ripples](#extensions)

Run sections 1-4 in order. Optional cells have switches set to `False`; enable only the workflows you need. All generated files go under a fresh `og_lab/tutorial_<id>/` folder. No external database or API credential is needed.

<a id="setup"></a>
## 1. Set up and inspect a crystal

`OgStructure` connects ASE builders with pymatgen's representation. Here `a=3.6` is a chosen cubic lattice parameter in angstrom, not an optimised value. ASE builds a one-atom primitive fcc cell by default; `cubic=True` gives the four-atom conventional cell.

The setup changes the working directory to an isolated output folder. Re-running it creates a fresh folder to avoid collisions with simulation outputs.

In [ ]:
from pathlib import Path
from uuid import uuid4
import os
import random
import numpy as np
from ase.build import bulk
from oganesson.ogstructure import OgStructure

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "examples/structures/MoS2.vasp").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the Oganesson repository.")
RUN = ROOT / "og_lab" / f"tutorial_{uuid4().hex[:8]}"
RUN.mkdir(parents=True)
os.chdir(RUN)
Path("og_lab").mkdir()  # xrd() expects this directory to exist.
random.seed(42)
np.random.seed(42)
copper = OgStructure(bulk("Cu", "fcc", a=3.6))
print("Outputs:", RUN)
print("Composition:", copper.structure.composition.reduced_formula)
print("Sites:", len(copper))
print("Volume (angstrom^3):", copper.structure.volume)
copper.structure.to(filename="Cu.cif", fmt="cif")

### Native fields: pymatgen- and ASE-shaped views

`OgStructure` keeps one internal array-based store and reads it two ways, without building an intermediate object each time: pymatgen-shaped fields (`.frac_coords`, `.cart_coords`, `.lattice_matrix`, `.a`/`.b`/`.c`/`.alpha`/`.beta`/`.gamma`, `.volume`, `.atomic_numbers`) and ASE-shaped fields (`.positions`, `.numbers`, `.symbols`, `.cell`, `.pbc`, `.get_cell_lengths_and_angles()`, `.get_all_distances(mic=True)`). `.structure` and `.to_ase()` remain available and build a genuine pymatgen `Structure` or ASE `Atoms` object on demand for methods not covered by these fields (symmetry, XRD and so on).

`get_all_distances(mic=True)` applies the minimum-image convention: with periodic boundaries, it reports the shortest image-to-image separation rather than the raw coordinate difference.

In [ ]:
print("Fractional coordinates:", copper.frac_coords)
print("Lattice matrix (angstrom):\n", copper.lattice_matrix)
print("a, b, c, alpha, beta, gamma:",
      copper.a, copper.b, copper.c, copper.alpha, copper.beta, copper.gamma)
print("ASE-style symbols:", copper.symbols)
print("Cell lengths and angles:", copper.get_cell_lengths_and_angles())

assert np.allclose(copper.frac_coords, copper.structure.frac_coords)
assert np.allclose(copper.positions, copper.to_ase().get_positions())

pair = OgStructure(bulk("Cu", "fcc", a=3.6).repeat((2, 1, 1)))
print("Pairwise distances with the minimum-image convention (angstrom):")
print(pair.get_all_distances(mic=True))

Read files with `OgStructure(file_name="Cu.cif")`. Use `.structure` for pymatgen operations and `.to_ase()` for ASE. Many methods modify the object in place; preserve a reference with `OgStructure(original.structure.copy())`. Use explicit `filename=` and `fmt=` when writing files.

<a id="preparation"></a>
## 2. Prepare alloys, surfaces and defects

### A substitutional alloy at fixed composition

A 4 x 4 x 4 primitive Cu supercell has 64 sites. The substitution dictionary specifies **atom counts**, not percentages, and must account for all sites being replaced. This creates an equiatomic Al-Cr-Ti-V candidate on an fcc template. A random configuration is neither a special quasirandom structure nor evidence that this alloy adopts the fcc phase.

In [ ]:
alloy = OgStructure(bulk("Cu", "fcc", a=3.6).repeat((4, 4, 4)))
counts = {"Al": 16, "Cr": 16, "Ti": 16, "V": 16}
assert sum(counts.values()) == len(alloy)
alloy.substitutions_random("Cu", counts)
alloy.structure.to(filename="AlCrTiV_fcc_candidate.cif", fmt="cif")
print(alloy.structure.composition)

### Inspect species order, bonds and composition class

`sort_species()` returns a new `OgStructure` with pymatgen's canonical species ordering; it does not modify `alloy` in place (`oganesson.genetic_algorithms.GA` relies on this when given `population=`). `get_bonds()` collects the distinct separations observed within a 4 angstrom neighbour search, keyed by atomic-number pair; `get_bonds_blocks()` runs the same search but keys by periodic-table column instead, coarsening chemically similar pairs together. `is_transition_metal()` checks whether every species in the structure is a transition metal.

In [ ]:
sorted_alloy = alloy.sort_species()
print("Original species (first 5):", [str(s) for s in alloy.structure.species[:5]])
print("Sorted species (first 5):", [str(s) for s in sorted_alloy.structure.species[:5]])

bonds_by_element_pair = alloy.get_bonds()
bonds_by_column_pair = alloy.get_bonds_blocks()
print("Element-pair bond keys:", list(bonds_by_element_pair.keys()))
print("Column-pair bond keys:", list(bonds_by_column_pair.keys()))
print("Is the alloy composed entirely of transition metals?", alloy.is_transition_metal())

For symmetry-distinct arrangements, `substitutions()` enumerates configurations instead of replacing sites randomly. Enumeration can grow rapidly with supercell size; the example below needs `bsym`.

In [ ]:
RUN_SUBSTITUTIONS_ENUMERATE = False  # Requires bsym.
if RUN_SUBSTITUTIONS_ENUMERATE:
    template = OgStructure(bulk("Cu", "fcc", a=3.6).repeat((2, 2, 2)))
    configurations = template.substitutions("Cu", {"Fe": 4, "Cu": 4})
    print("Distinct configurations:", len(configurations))
    for index, candidate in enumerate(configurations):
        candidate.structure.to(filename=f"FeCu_config_{index:03d}.cif", fmt="cif")

### A Li adsorption candidate on MoS2

The placement routine searches geometrically and can fail. Inspect surface orientation, vacuum and atom separations before relaxation. This step does not calculate adsorption energy or identify the most stable site.

In [ ]:
surface = OgStructure(file_name=str(ROOT / "examples/structures/MoS2.vasp"))
adsorbed = surface.add_atom_to_surface("Li")
if adsorbed is False:
    print("No adsorption candidate found with these settings.")
else:
    adsorbed.structure.to(filename="MoS2_Li.vasp", fmt="poscar")

### Passivate a dangling bond

`passivate(atom, passivating_atom_symbol, d=2.5)` averages the vectors from `atom` to its neighbours within 3.5 angstrom, places a new atom at distance `d` along the direction opposite that average, and appends it. It caps one atom index at a time and does not identify dangling bonds automatically.

In [ ]:
passivated = OgStructure(surface.structure.copy())
passivated.passivate(0, "H", d=1.5)
print("Atoms before:", len(surface), "after:", len(passivated))
passivated.structure.to(filename="MoS2_passivated.cif", fmt="cif")

### Adsorb a molecule

`add_molecule_to_surface()` places a whole ASE `Atoms` object above the surface (levelling it first with `center()`/`zero_z()`), retrying with a random in-plane offset whenever a candidate placement fails a distance check. It returns `False` after `max_trials` failed attempts.

In [ ]:
from ase.build import molecule

water = molecule("H2O")
water.cell = [10, 10, 10]
molecule_adsorbed = surface.add_molecule_to_surface(water, displacement=2.5, max_trials=200)
if molecule_adsorbed is False:
    print("No adsorption placement found with these settings.")
else:
    print("Atoms before:", len(surface), "after:", len(molecule_adsorbed))
    molecule_adsorbed.structure.to(filename="MoS2_H2O.vasp", fmt="poscar")

### Scan adsorption sites (optional)

`adsorption_scanner()` samples surface positions. Its last returned structure is an **overview containing all proposed adsorbates**, not an individual adsorption configuration. Save it separately from candidates intended for energy comparisons.

In [ ]:
RUN_ADSORPTION_SCAN = False
if RUN_ADSORPTION_SCAN:
    from ase.build import fcc111
    gold = fcc111("Au", size=(2, 2, 3), a=4.08, vacuum=10.0)
    scanned = OgStructure(gold).adsorption_scanner("O")
    for index, candidate in enumerate(scanned[:-1]):
        candidate.structure.to(filename=f"Au_O_{index:03d}.cif", fmt="cif")
    scanned[-1].structure.to(filename="Au_O_site_overview.cif", fmt="cif")
    print("Individual candidates:", len(scanned) - 1)

### Place an interstitial (optional)

`add_interstitial()` searches for a geometrical void and modifies the host on success. It returns `False` if its distance check rejects the candidate. H in Cu illustrates placement only: defect charge, formation energy and site stability require further calculations. The default grid search can take time.

In [ ]:
RUN_INTERSTITIAL = False
if RUN_INTERSTITIAL:
    host = OgStructure(bulk("Cu", "fcc", a=3.6).repeat((3, 3, 3)))
    defect = host.add_interstitial("H")
    if defect is False:
        print("No interstitial accepted; inspect the host and distance settings.")
    else:
        defect.structure.to(filename="Cu_H_interstitial.cif", fmt="cif")

### Geometric operations

These methods edit the current structure in place and return `self`, so calls can be chained. `translate(v, frac_coords=False)` shifts every atom by the same vector; because pymatgen's `translate_sites()` wraps atoms back into the cell by default, the recovered displacement for a given atom in a repeated cell can differ from `v`, so the example below uses a single-atom cell where atom 0 is not wrapped. `center(about_atom=...)`/`centerXY(i)` recentre on an atom (`centerXY` only in the surface plane). `repeat(scaling_matrix)` builds a supercell (equivalent to `make_supercell`). `scale(s)` rescales the lattice and moves atoms with it, keeping fractional coordinates fixed; `scale_lattice_only(s)` rescales only the lattice, keeping cartesian coordinates fixed, which changes the fractional coordinates instead. `set_positions(positions)` replaces cartesian coordinates directly. `zero_z()`/`zero(axis=...)` shift all atoms so the minimum coordinate along that axis is zero, useful before stacking or plotting a slab.

In [ ]:
mobile = OgStructure(bulk("Cu", "fcc", a=3.6))
mobile.translate([0.2, 0.0, 0.0])
print("Cartesian shift for a single-atom cell:", mobile.structure.cart_coords[0])

pair_geo = OgStructure(bulk("Cu", "fcc", a=3.6).repeat((2, 1, 1)))
pair_geo.center(about_atom=0)
print("Fractional coordinates after centring on atom 0:", pair_geo.frac_coords)

slab_geo = OgStructure(bulk("Cu", "fcc", a=3.6).repeat((2, 1, 1)))
slab_geo.centerXY(0)
print("XY coordinates of atom 0 after centerXY:", slab_geo.structure.cart_coords[0][:2])

supercell = OgStructure(bulk("Cu", "fcc", a=3.6))
supercell.repeat([2, 2, 2])
print("Atoms after repeat([2, 2, 2]):", len(supercell))

strained = OgStructure(bulk("Cu", "fcc", a=3.6))
volume_before = strained.structure.volume
strained.scale(1.05)
print("Volume ratio after scale(1.05):", strained.structure.volume / volume_before)

lattice_only = OgStructure(bulk("Cu", "fcc", a=3.6))
coords_before = lattice_only.structure.cart_coords.copy()
lattice_only.scale_lattice_only([1.0, 1.0, 1.2])
print("Cartesian coordinates unchanged by scale_lattice_only:",
      np.allclose(coords_before, lattice_only.structure.cart_coords))
print("New c lattice parameter:", lattice_only.structure.lattice.c)

repositioned = OgStructure(bulk("Cu", "fcc", a=3.6).repeat((2, 1, 1)))
new_positions = repositioned.structure.cart_coords.copy()
new_positions[0] += [0.05, 0.0, 0.0]
repositioned.set_positions(new_positions)
print("Atom 0 after set_positions:", repositioned.structure.cart_coords[0])

slab_like = OgStructure(bulk("Cu", "fcc", a=3.6).repeat((2, 2, 3)))
slab_like.zero_z()
print("Minimum z after zero_z():", slab_like.structure.cart_coords[:, 2].min())

<a id="analysis"></a>
## 3. Calculate RDF and XRD

### Radial distribution function: which separations occur?

`get_rdf()` returns `(g_r, distances)`. Distances and `rmax` are in angstrom. The periodic cell must enclose a sphere of radius `rmax`; this alloy supercell accommodates 4 angstrom. Add `elements=[13, 13]` for an Al-Al partial RDF. The volume normalisation is intended for bulk structures; slab vacuum changes that normalisation.

In [ ]:
import matplotlib.pyplot as plt

g_r, distances = alloy.get_rdf(rmax=4.0, nbins=100)
fig, ax = plt.subplots()
ax.plot(distances, g_r)
ax.set(xlabel="r (angstrom)", ylabel="g(r)", title="Unrelaxed alloy: total RDF")
fig.savefig("alloy_rdf.png", bbox_inches="tight")
plt.show()

### X-ray diffraction: compare periodic structures

`xrd()` returns a pymatgen diffraction pattern and saves a plot under `og_lab/` inside the tutorial output folder. `pattern.x` contains 2-theta angles in degrees; `pattern.y` contains intensities. This is an ideal-structure calculation. Experimental comparisons also require attention to sample and instrumental effects.

In [ ]:
alloy.structure_tag = "AlCrTiV_fcc_candidate"
pattern = alloy.xrd(two_theta_range=(10, 90))
print("2-theta (degrees):", pattern.x[:5])
print("Intensities:", pattern.y[:5])

### Composition-based quantities

`get_atom_count(atom)` counts sites of one element; `calculate_molecular_mass()` sums atomic masses over all sites (atomic mass units, not normalised per formula unit); `calculate_theoretical_capacity(charge_carrier, n=None)` applies Q = nF / (3600 * M_w) * 1000 to give a theoretical specific capacity in mAh/g, where `F` is Faraday's constant and, by default, `n = get_atom_count(charge_carrier)`. This treats every atom of the charge carrier as extractable and ignores the host's actual electrochemical behaviour; for pure Li metal it reproduces the textbook value of about 3861 mAh/g.

In [ ]:
lithium = OgStructure(bulk("Li", "bcc", a=3.49))
print("Li atom count:", lithium.get_atom_count("Li"))
print("Molecular mass (amu):", lithium.calculate_molecular_mass())
print("Theoretical specific capacity (mAh/g):", lithium.calculate_theoretical_capacity("Li"))

<a id="descriptors"></a>
## 4. Build descriptors and a feature dataset

Descriptors provide numerical inputs to a property model. BACD combines elemental-property statistics, geometry and a space-group encoding. Tabulated elemental properties are not predictions of compound properties; the implementation replaces missing elemental values with zero. `SymmetryFunctions` aggregates radial neighbour functions, including atomic-number-weighted terms.

Use consistent cell conventions across your dataset: a structure-level vector does not necessarily remove cell-size dependence.

In [ ]:
from oganesson.descriptors import BACD, SymmetryFunctions

bacd_features = np.asarray(BACD(copper).describe(), dtype=float)
radial_features = np.asarray(SymmetryFunctions(copper).describe(), dtype=float)
print("BACD shape:", bacd_features.shape)
print("SymmetryFunctions shape:", radial_features.shape)
print("Finite BACD values:", np.isfinite(bacd_features).all())

### Check one translation

Invariance means the representation stays unchanged under a specified transformation; equivariance means it transforms correspondingly. This comparison checks one translation, not rotations, atom reordering or all structures. Chemical substitution changes the material and should generally change its features.

In [ ]:
translated = copper.structure.copy()
translated.translate_sites(list(range(len(translated))), [0.17, 0.23, 0.31],
                           frac_coords=True, to_unit_cell=True)
shifted_features = np.asarray(BACD(translated).describe(), dtype=float)
print("Unchanged after translation:",
      np.allclose(bacd_features, shifted_features, rtol=1e-6, atol=1e-4))

### Optional DScribe features

Install `dscribe` before enabling this cell. A sine matrix describes periodic structures. Across a dataset, fix `n_atoms_max` to at least the largest atom count to keep vector lengths consistent.

Other exported wrappers are `DscribeACSF`, `DScribeSOAP`, `DScribeCoulombMatrix` and `DScribeEwaldSumMatrix`; they require compatible DScribe APIs. ROSA separately requires GPAW and its datasets.

In [ ]:
RUN_DSCRIBE = False
if RUN_DSCRIBE:
    from oganesson.descriptors import DScribeSineMatrix
    sine_features = DScribeSineMatrix(copper, n_atoms_max=4).describe()
    print("Feature shape:", np.asarray(sine_features).shape)

### Assemble a local feature table

These three ideal Cu cells demonstrate data organisation. They are related, unlabelled structures, too few to train or evaluate a useful property model. Each row retains its structure identifier; the CIF files preserve the geometries.

In [ ]:
import pandas as pd

records = []
for a in (3.5, 3.6, 3.7):
    sample = OgStructure(bulk("Cu", "fcc", a=a))
    sample_id = f"Cu_a{a:.1f}"
    sample.structure.to(filename=f"{sample_id}.cif", fmt="cif")
    vector = np.asarray(BACD(sample).describe(), dtype=float)
    records.append({"structure_id": sample_id,
                    **{f"bacd_{i:03d}": value for i, value in enumerate(vector)}})
features = pd.DataFrame(records).set_index("structure_id")
if not np.isfinite(features.to_numpy()).all():
    raise ValueError("Investigate non-finite features before modelling.")
features.to_csv("bacd_features.csv")
print(features.shape)
features.iloc[:, :6]

For property prediction, replace these examples with a curated collection and join reference targets by `structure_id`. Record target units (for example, eV/atom for formation energy), calculation settings and data provenance. Keep related structures together when separating training and evaluation data. Fit preprocessing on training data only and report held-out errors against a baseline. This example makes no claim about predictive accuracy.

<a id="relaxation"></a>
## 5. Relax structures and prepare NEB images

### Relaxation with a machine-learned potential (optional)

`model="diep"` loads the bundled potential and is the current default. `model="m3gnet"` explicitly requests MatGL's named M3GNet model and requires the `matgl` extra. Other strings are passed to the DIEP loader. Check the potential's coverage and accuracy for your chemistry.

`relax()` modifies the structure and stores `total_energy` and `trajectory`. This example holds the cell fixed and requests a force threshold of 0.05 eV/angstrom. Reaching the step limit does not establish convergence.

In [ ]:
RUN_RELAXATION = False
if RUN_RELAXATION:
    relaxed = OgStructure(copper.structure.copy())
    relaxed.relax(model="diep", relax_cell=False, steps=100, fmax=0.05)
    relaxed.structure.to(filename="Cu_relaxed.cif", fmt="cif")
    print("Total energy (eV):", relaxed.total_energy)

### Recover a displacement vector

`get_delta_vector(structure)` returns, for each atom, the minimum-image cartesian displacement from the current structure to `structure` — the same shortest-vector logic used internally by `get_rdf()` and by `get_all_distances(mic=True)`, applied atom-by-atom rather than pairwise. This is useful for comparing two images of the same structure, for example two steps of a trajectory, without constructing an interpolation path.

In [ ]:
endpoint_initial = OgStructure(bulk("Cu", "fcc", a=3.6).repeat((2, 2, 2)))
endpoint_final = OgStructure(endpoint_initial.structure.copy())
endpoint_final.translate([0.05, 0.0, 0.0])
displacement = endpoint_initial.get_delta_vector(endpoint_final.structure)
print("Displacement of atom 0 (angstrom):", displacement[0])
print("Largest displacement magnitude (angstrom):", np.abs(displacement).max())

### Interpolate an explicit vacancy hop (optional)

This geometry example builds a Cu vacancy hop: remove one atom from a 32-atom conventional supercell, then move a neighbouring atom into the vacancy. For research, first relax both endpoints with consistent cells, atom ordering and settings.

`num_images=5` means five intermediate images plus two endpoints. Pass `model=None` explicitly for linear interpolation with the minimum-image convention; the helper's default is not `None`. The current `frozen_atoms` argument is unused and does not impose constraints.

In [ ]:
RUN_NEB = False
if RUN_NEB:
    perfect = OgStructure(bulk("Cu", "fcc", a=3.6, cubic=True).repeat((2, 2, 2)))
    initial = perfect.structure.copy()
    vacancy_position = initial[0].frac_coords.copy()
    initial.remove_sites([0])
    final = initial.copy()
    final.replace(0, "Cu", coords=vacancy_position, coords_are_cartesian=False)
    initial = OgStructure(initial)
    final = OgStructure(final)
    initial.generate_neb_images(final, frozen_atoms=[], num_images=5, model=None,
                                folder_tag=f"Cu_vacancy_{uuid4().hex[:8]}")

Files appear in `neb_path_Cu_vacancy_<id>/00/POSCAR` through `06/POSCAR`, plus `00.vasp` through `06.vasp` previews. Supply VASP calculation inputs and convergence settings separately. These geometries alone do not establish a migration barrier.

### Automatic same-species path enumeration

For your own Li-containing host, the alternative is `host.generate_neb("Li", r=3.0, num_images=5, model="diep")`. The Li3PO4 and LGPS CIFs referenced in older examples are not bundled.

`r` is a same-species neighbour search radius in angstrom, not an ionic radius or a universally sufficient cutoff. This routine removes one atom to form vacancy-hop endpoints, then independently relaxes every interpolated image. That is not coupled NEB optimisation and may erase the intended path. It writes POSCAR-format files named `00`, `01`, etc. directly inside `neb_path_<i>_<j>/`, not VASP image subdirectories. Inspect and reorganise these files before a VASP NEB run. Use a fresh directory to avoid folder collisions.

<a id="dynamics"></a>
## 6. Run molecular dynamics (optional)

`simulate()` uses a machine-learned potential: this is ML molecular dynamics, not ab initio molecular dynamics (AIMD). Supply your own relaxed periodic structure in `MD_INPUT`; no electrolyte structure is bundled.

The example requests NVT at 1000 K with a 1 fs time step for 1000 steps (1 ps). This short run checks the workflow, not a converged diffusion coefficient. Choose temperature, equilibration, cell size and sampling duration for your material. Inspect the thermodynamic log and trajectory before diffusion fitting.

In [ ]:
RUN_MD = False
MD_INPUT = ROOT / "my_inputs" / "relaxed_electrolyte.cif"
if RUN_MD:
    if not MD_INPUT.is_file():
        raise FileNotFoundError(f"Provide a relaxed structure at {MD_INPUT}")
    dynamics = OgStructure(file_name=str(MD_INPUT))
    dynamics.simulate(model="diep", ensemble="nvt", temperature=1000,
                      timestep=1, steps=1000, loginterval=1,
                      folder_tag=f"electrolyte_{uuid4().hex[:8]}")
    print("Trajectory:", dynamics.trajectory_file)
    print("Log:", dynamics.log_file)

Tracer diffusion analysis uses mean-squared displacements (MSDs). Select an equilibrated interval exhibiting diffusive behaviour. Keep `loginterval=1`: the wrapper passes the integration time step to the analysis without accounting for larger frame-saving intervals. `ignore_n_images` counts saved frames.

The method returns the diffusivity package's results and saves an MSD plot beside the trajectory. Check that package's output units before reporting values. Sampling uncertainty, rare hops and finite-size effects need further analysis.

In [ ]:
RUN_DIFFUSIVITY = False
if RUN_DIFFUSIVITY:
    if not RUN_MD or "dynamics" not in globals():
        raise RuntimeError("Run and inspect the MD example first.")
    coefficients = dynamics.calculate_diffusivity(
        calculation_type="tracer", axis="all", ignore_n_images=100)
    print("Diffusivity analysis output:", coefficients)

<a id="search"></a>
## 7. Search structures with a genetic algorithm (optional)

A genetic algorithm generates and modifies candidates, relaxing them with an interatomic potential. Here the composition is fixed at four Na and four H atoms. Lower total energy gives higher fitness (`raw_score = -energy`); these comparisons require fixed composition.

The constructor creates an initial population and database. The first `evolve()` call relaxes that population; subsequent calls generate offspring. These small settings illustrate the API and do not establish the NaH ground state. Check the model, volume and cell bounds for your chemistry, repeat searches and separately assess competing phases.

In [ ]:
RUN_GA = False
if RUN_GA:
    from oganesson.genetic_algorithms import GA
    ga = GA(species=["Na"] * 4 + ["H"] * 4, population_size=10,
            box_volume=240, rmax=10, steps=100, model="diep")
    ga.evolve(num_offsprings=5)  # Relax the initial population.
    for _ in range(2):
        ga.evolve(num_offsprings=5)
    print("Search outputs:", ga.path)

Outputs under `og_lab/ga_<id>/` include `ga.db`, relaxed CIF files and population trajectories after offspring evolution. Short relaxation limits can leave candidates unconverged. Re-relax promising structures and compare with reference calculations.

The API also accepts `GA(population=candidates)`. That path is not demonstrated here: the constructor first calls `sort_species()` on each entry and reassigns the result back into `population`. `sort_species()` returns a new `OgStructure` with a pymatgen-sorted species order rather than modifying the original in place; ASE's GA machinery needs this consistent ordering across the population.

<a id="extensions"></a>
## 8. Extract VASP data and construct ripples (optional)

### Extract an existing VASP trajectory

Supply your own `OUTCAR` and matching `POSCAR`; neither is bundled. The POSCAR identifies species explicitly. The extractor writes energies, structures, forces and stresses to JSON. Check arrays and conventions against the original calculation before using them as labels. Directory arguments need trailing separators because the implementation concatenates strings.

In [ ]:
RUN_VASP_EXTRACTION = False
if RUN_VASP_EXTRACTION:
    from oganesson.io.vasp import Outcar
    vasp_directory = ROOT / "my_inputs" / "vasp_md"
    for name in ("OUTCAR", "POSCAR"):
        if not (vasp_directory / name).is_file():
            raise FileNotFoundError(vasp_directory / name)
    outcar = Outcar(outcar_directory=str(vasp_directory) + os.sep,
                    outcar_file="OUTCAR", poscar_file="POSCAR")
    outcar.write_md_data(file_name="vasp_md.json", path=str(RUN) + os.sep)

### Construct a ripple in a two-dimensional material

Install `sympy` first; it is the only extra dependency `create_ripple()` needs beyond the base install. `create_ripple()` repeats the sheet along the selected axis and imposes a wave. In this API, **`strain` is target length divided by original length**: `strain=0.95` means 5% shortening, not 95% strain.

`relax=False` produces a geometric construction. For finite-thickness layers such as MoS2, inspect the geometry and use staged relaxation (`relax=True`, `model="diep"`) when appropriate. `steps` then controls intermediate deformation stages. To save intermediates, create a directory and pass `write_intermediate=True` and `intermediates_folder="that_directory"`.

In [ ]:
RUN_RIPPLE = False
if RUN_RIPPLE:
    sheet = OgStructure(file_name=str(ROOT / "examples/structures/MoS2.vasp"))
    ripple = sheet.create_ripple(axis="x", units=10, strain=0.95, relax=False)
    ripple.structure.to(filename="MoS2_ripple_unrelaxed.cif", fmt="cif")

### Wrap a chain into a helix

`create_helix(length, amplitude)` returns a new `OgStructure` with `y` and `z` displaced sinusoidally as a function of `x`, using `length` as the helical period. It is a geometric construction, not a relaxed or physically stable conformation, and acts on the cartesian `x` coordinate regardless of whether the lattice is actually periodic along that axis.

In [ ]:
chain = OgStructure(bulk("Cu", "fcc", a=3.6).repeat((4, 1, 1)))
helix = chain.create_helix(length=chain.structure.lattice.a, amplitude=0.5)
print("Atoms preserved:", len(chain) == len(helix))
helix.structure.to(filename="Cu_chain_helix.cif", fmt="cif")

### Fracture a bulk region under strain (optional)

`fracture()` incrementally strains the cell along one axis while relaxing with a machine-learned potential (`model="diep"` by default), pulling terminal atoms apart and freezing them once tagged. Its `strain` argument is the same target-length-over-original-length convention as `create_ripple()`. It can write intermediate structures via `write_intermediate=True`/`intermediates_folder=...`. This needs the same `diep` potential as `relax()`.

In [ ]:
RUN_FRACTURE = False  # Requires diep.
if RUN_FRACTURE:
    block = OgStructure(bulk("Cu", "fcc", a=3.6, cubic=True).repeat((1, 1, 6)))
    block.fracture(strain=1.5, axis="z", steps=20, model="diep",
                   write_intermediate=True, intermediates_folder="fracture_steps")
    block.structure.to(filename="Cu_fractured.cif", fmt="cif")

### Sample an approximate electron density

`generate_density(ngrid, K, pbc_mode, nx, ny, nz, fmt, out)` builds a periodic numerical extended-Hueckel-style density and samples it on a real-space grid of shape `(nx, ny, nz)`, writing a Gaussian cube (`fmt="cube"`) or a VESTA/VASP-style CHGCAR (`fmt="chgcar"`) file. `ngrid` sets the reciprocal-space sampling used to build the density and `K` is a decay parameter for the underlying atomic-like orbitals. This is a fast, qualitative visualisation tool, not a self-consistent DFT density.

In [ ]:
copper.generate_density(ngrid=(24, 24, 24), K=1.75, pbc_mode="minimum",
                        nx=24, ny=24, nz=24, fmt="chgcar", out="Cu_density.CHGCAR")
print("Density file written:", Path("Cu_density.CHGCAR").stat().st_size, "bytes")

### Build a commensurate bilayer cell (optional)

`commensurate([other_structure], MAX, error, vacuum)` searches, up to a repeat count of `MAX` along `a` and `b`, for integer supercells of `self` and `other_structure` whose in-plane lattice parameters match within `error` percent; both lattices must be cubic. On success it returns three structures: the stacked bilayer with `vacuum` angstrom of separation between the two films, and the two matched supercells individually. It returns `(None, None, None)` if no match is found within `MAX`; raising `MAX` searches further but grows combinatorially.

In [ ]:
RUN_COMMENSURATE = False  # Slow for larger MAX; disabled by default for safety.
if RUN_COMMENSURATE:
    film_a = OgStructure(bulk("Cu", "fcc", a=3.6, cubic=True))
    film_b = OgStructure(bulk("Ni", "fcc", a=3.5, cubic=True))
    combined, matched_a, matched_b = film_a.commensurate(
        [film_b.structure], MAX=5, error=5, vacuum=10)
    if combined is None:
        print("No commensurate match found within MAX.")
    else:
        print("Atoms in the combined cell:", len(combined))
        combined.structure.to(filename="Cu_Ni_commensurate.cif", fmt="cif")

## Record a reproducible calculation

Save initial structures, random seeds, model identity, package versions and calculation settings with your results. State whether conclusions come from generated geometry, potential predictions or validated reference calculations. The cell below records package versions without loading extra models. These random seeds do not guarantee deterministic behaviour across all backends.

In [ ]:
import json
from importlib.metadata import PackageNotFoundError, version

versions = {}
for package in ("oganesson", "ase", "pymatgen", "numpy", "pandas", "diep", "matgl", "dscribe"):
    try:
        versions[package] = version(package)
    except PackageNotFoundError:
        versions[package] = "not installed"
(RUN / "package_versions.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")
print("Tutorial outputs:", RUN)
versions